In [ ]:
##############################
# DOWNSAMPLING
##############################

import json
import os
import numpy as np
from scipy.signal import argrelextrema
import time as timer

# Target directory for downsampling
fixture_dir = "./path/to/fixtures"

anonymized_time = np.array([])
anonymized_voltage = np.array([])
anonymized_current = np.array([])

downsampled_time = np.array([])
downsampled_current = np.array([])
downsampled_voltage = np.array([])

# Downsample to 10,000 points
num_files = len(os.listdir(fixture_dir))
n = 10000 // num_files

for filename in os.listdir(fixture_dir):

    start_time = timer.time()
    
    if filename.endswith(".json"):
        with open(os.path.join(fixture_dir, filename), 'r') as f:
            fixtures = json.load(f)

            time = []
            voltage = []
            current = []

            for obj in fixtures:
                fields = obj['fields']
                data = fields['data']
                time.append(data['time'])
                voltage.append(data['anonymized_voltage'])
                current.append(data['anonymized_current'])

            # Downsample
            time = np.array(time)
            anonymized_time = np.concatenate([anonymized_time, time])
            voltage = np.array(voltage)
            anonymized_voltage = np.concatenate([anonymized_voltage, voltage])
            current = np.array(current)
            anonymized_current = np.concatenate([anonymized_current, current])

            # Detect peaks and valleys in voltage data
            voltage_peaks = argrelextrema(voltage, np.greater, order=5)[0]
            voltage_valleys = argrelextrema(voltage, np.less, order=5)[0]
            voltage_critical = np.unique(np.concatenate([voltage_peaks, voltage_valleys]))

            # Detect asymptotes in current and voltage data
            voltage_diff = np.abs(np.diff(voltage))
            current_diff = np.abs(np.diff(current))

            voltage_asymptotes = np.where(voltage_diff > np.percentile(voltage_diff, 95))[0] + 1
            current_asymptotes = np.where(current_diff > np.percentile(current_diff, 99))[0] + 1

            # Combine critical indices
            voltage_critical = np.unique(np.concatenate([voltage_critical, voltage_asymptotes]))
            current_critical = current_asymptotes
            all_critical = np.unique(np.concatenate([voltage_critical, current_critical, [0, len(time) - 1]]))

            weights = np.zeros(len(time))
            weights[1:] += voltage_diff
            weights[1:] += current_diff

            if len(all_critical) > 0:
                sorted_indices = np.argsort(weights[all_critical])[-n:]
                keep_indices = all_critical[sorted_indices]
            else:
                keep_indices = np.linspace(0, len(time)-1, n, dtype=int)
            
            if len(keep_indices) < n and len(time) > n:
                keep_indices = np.linspace(0, len(time)-1, n, dtype=int)

            # Extract downsampled time
            downsampled_time = np.concatenate([downsampled_time, time[keep_indices]])
            downsampled_voltage = np.concatenate([downsampled_voltage, voltage[keep_indices]])
            downsampled_current = np.concatenate([downsampled_current, current[keep_indices]])

            end_time = timer.time()
            print(f"Processed {filename} in {end_time - start_time} seconds")


In [16]:
##############################
# VISUAL COMPARISONS
##############################
import matplotlib.pyplot as plt

In [ ]:
# Voltage Comparison
plt.figure(figsize=(12, 6))
plt.scatter(anonymized_time, anonymized_voltage, label='Original Voltage', color='lightblue', alpha=0.5)
plt.scatter(downsampled_time, downsampled_voltage, label="Olive Tree Voltage", color='blue', alpha=1, s=10)
plt.legend()
plt.show()

In [ ]:
# Current Comparison
plt.figure(figsize=(12, 6))
plt.scatter(anonymized_time, anonymized_current, label='Original Current', color='lightblue', alpha=0.5)
plt.scatter(downsampled_time, downsampled_current, label="Olive Tree Current", color='blue', alpha=1, s=10)
plt.legend()
plt.show()